In [ ]:
# General imports
import numpy as np
import matplotlib.pyplot as plt

# cloelib imports
from cloelib.cosmology.camb_cosmology import CAMBBackground

## Cosmology: Background

This code snippet initializes and computes the cosmological background.

In [2]:
zs = np.linspace(1e-4, 3, 1000)

# The arguments of the Background functions follow the cosmology.API
background = CAMBBackground(H0=70.0, 
                            Omega_cdm0=0.248684,
                            Omega_b0=0.05, 
                            w0=-1, 
                            wa=0, 
                            Omega_k0 = 0.0, 
                            ns = 0.96, 
                            As = 2e-9,
                            mnu = 0.06,
                            gamma_MG = 0.545,
                            N_mnu = 1)

## Construct perturbations directly from external linear and nonlinear power spectra

### In this example, the external power spectra come from CosmoSIS. 
CosmoSIS uses h-based units (h/Mpc and (Mpc/h)^3), so we convert to physical units (Mpc^-1, Mpc^3) before passing them in.

In [ ]:
# Local path to external output files
input_path_external_data = '/Users/your_local_path/cosmosis_data/'
input_path_external_matter_power_lin = input_path_external_data + 'matter_power_lin/'
input_path_external_matter_power_nl = input_path_external_data + 'matter_power_nl/'
z_external = np.loadtxt(input_path_external_matter_power_lin + 'z.txt')
k_external = np.loadtxt(input_path_external_matter_power_lin + 'k_h.txt')*background.h
p_k_lin_external = np.loadtxt(input_path_external_matter_power_lin + 'p_k.txt')/(background.h**3)
p_k_nl_external = np.loadtxt(input_path_external_matter_power_nl + 'p_k.txt')/(background.h**3)

In [4]:
from tabulated_pk_interpolator import TabulatedMatterPowerInterpolator

p_k_lin_external_interp = TabulatedMatterPowerInterpolator(z_external, k_external, p_k_lin_external)
p_k_nl_external_interp = TabulatedMatterPowerInterpolator(z_external, k_external, p_k_nl_external)

In [5]:
from tabulated_perturbations import TabulatedPerturbations
# Perturbations built directly from the external matter_power_lin/nl tables
# loaded above, instead of computed by CAMB - satisfies the same
# cloelib.cosmology.cosmology.Perturbations protocol as
# CAMBLinearPerturbations/CAMBNonLinearPerturbations.
linear_perturbations_external = TabulatedPerturbations(
    background, z_external, k_external, p_k_lin_external,
)
nonlinear_perturbations_external = TabulatedPerturbations(
    background, z_external, k_external, p_k_nl_external,
    linearperturbations=linear_perturbations_external,
)